# Wine Quality Predictive Analytics (Mini-Project)
### Core Syllabus Implementation: EDA, Data Preparation, Regression, LDA & QDA

**Course / Topic:** Predictive Analytics / Predictive Modeling  
**Dataset:** Cortez et al. (2009), UCI Machine Learning Repository (Red Wine Quality, 1,599 instances, 11 physicochemical features)  
**Primary Objectives:**
1. **Data Preparation:** Data auditing, handling duplicates, checking missing values, outlier inspection, standard scaling, and train-test splits.
2. **Exploratory Data Analysis (EDA):** Univariate distributions, bivariate boxplots, correlation heatmap, and driver identification.
3. **Regression Analysis:** Simple Linear Regression, Multiple Linear Regression, and L2 Regularized Ridge Regression to predict numeric sensory quality scores (3 to 8).
4. **Discriminant Analysis & Classification:** Linear Discriminant Analysis (LDA) and Quadratic Discriminant Analysis (QDA) alongside Logistic Regression to predict high-quality wines ($\ge 7$).


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so src modules can be imported directly
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import classification_models as clf
from src import eda
from src import regression_models as reg
from src.data_preprocessing import (
    FEATURES,
    TARGET,
    BINARY_TARGET,
    load_wine_data,
    get_data_overview,
    quality_check,
    iqr_outlier_summary,
    clean_data,
    add_binary_target,
    split_regression,
    split_classification,
)

# Configure styling
eda.set_style()
pd.set_option("display.max_columns", 15)
print("Environment and modules successfully initialized.")


---
## 1. Data Preparation and Cleaning

In predictive analytics, data preparation ensures data integrity, prevents data leakage between training and testing sets, and validates assumptions before modeling.


In [ ]:
# Load raw dataset
raw_df = load_wine_data()
print(f"Dataset Dimensions: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
raw_df.head()


In [ ]:
# Check data types and verify missing values
overview = get_data_overview(raw_df)
health = quality_check(raw_df)
print("Quality Check Summary:", health)
overview


In [ ]:
# Outlier analysis using the 1.5 * IQR rule
outliers_df = iqr_outlier_summary(raw_df)
outliers_df[["IQR", "Lower Fence", "Upper Fence", "Outlier Count", "Outlier Pct (%)"]]


### Cleaning Decisions:
1. **Missing Values:** There are 0 missing values in the UCI dataset; no imputation is needed.
2. **Duplicate Rows:** 240 exact duplicate rows are identified. We remove them to prevent **data leakage** (the same sample appearing in both training and test sets, artificially inflating test scores).
3. **Outliers:** 1.5 * IQR outliers are observed (e.g., in residual sugar and chlorides). We **retain** these outliers because they reflect genuine physical wine variations rather than measurement errors.
4. **Binary Target Formulation:** We define `high_quality = 1` for wines with `quality >= 7` (premium rating) and `0` otherwise (< 7).


In [ ]:
# Clean data and add binary target
df, clean_log = clean_data(raw_df)
df = add_binary_target(df)
print("Cleaning Log:", clean_log)
print(f"Cleaned dataset shape: {df.shape}")
df[[TARGET, BINARY_TARGET]].head()


In [ ]:
# Train-test splits (80% train, 20% test with random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = split_regression(df)
X_train_c, X_test_c, y_train_c, y_test_c = split_classification(df)

print(f"Regression Split:     Train = {len(X_train_r)}, Test = {len(X_test_r)}")
print(f"Classification Split: Train = {len(X_train_c)}, Test = {len(X_test_c)} (Stratified)")


---
## 2. Exploratory Data Analysis (EDA)

EDA allows us to examine the distributions of predictors, identify the shape of the target variable, and understand the correlation structure.


In [ ]:
# Target Distribution (Continuous Score & Binary Class)
fig = eda.plot_target_distribution(df)
plt.show()


In [ ]:
# Feature Distributions (Histogram + KDE with Skewness)
fig = eda.plot_feature_distributions(df)
plt.show()


In [ ]:
# Outlier Inspection via Boxplots
fig = eda.plot_feature_boxplots(df)
plt.show()


In [ ]:
# Correlation Heatmap
fig = eda.plot_correlation_heatmap(df)
plt.show()


In [ ]:
# Key Driver Analysis: Alcohol (+) and Volatile Acidity (-)
fig = eda.plot_top_features_vs_quality(df)
plt.show()


In [ ]:
# Correlation of all features with Quality
corr_series = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
corr_table = pd.DataFrame({"Pearson r with Quality": corr_series.round(4)})
corr_table


---
## 3. Regression Analysis (Predicting Numeric Quality)

We model wine quality as a continuous score (3 to 8).
We evaluate:
- **Baseline (Mean):** Predicts the mean training quality.
- **Simple Linear Regression:** Univariate model using `alcohol` (the strongest positive predictor).
- **Multiple Linear Regression (MLR):** Fits a hyperplane using all 11 standardized features:
  $$y = \beta_0 + \sum_{j=1}^{11} \beta_j X_j + \epsilon$$
- **Ridge Regression:** L2 regularized regression that penalizes large coefficient magnitudes:
  $$\min_{\beta} \left\{ \sum_{i=1}^n (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^p \beta_j^2 \right\}$$


In [ ]:
# Build and evaluate regression models
reg_models = reg.build_regressors()
reg_models, reg_results, reg_preds = reg.fit_and_evaluate(
    reg_models, X_train_r, X_test_r, y_train_r, y_test_r
)
reg_results.round(4)


In [ ]:
# Standardized Coefficients of Multiple Linear Regression
mlr_coefs = reg.get_linear_coefficients(reg_models["Multiple Linear Regression"])
print("Feature Coefficients (Relative Importance):")
display(mlr_coefs.round(4))

# Visualize coefficients
plt.figure(figsize=(9, 4.5))
colors = ["#2a78d6" if c > 0 else "#eb6834" for c in mlr_coefs["Standardized_Coefficient"]]
plt.barh(mlr_coefs["Feature"], mlr_coefs["Standardized_Coefficient"], color=colors)
plt.axvline(0, color="#8a8984", ls="--")
plt.title("Standardized Coefficients (Multiple Linear Regression)", fontweight="bold")
plt.xlabel("Coefficient Value (Standardized Effect on Quality)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Diagnostic Plot 1: Actual vs Predicted Quality
fig = reg.plot_actual_vs_predicted(y_test_r, reg_preds)
plt.show()


In [ ]:
# Diagnostic Plot 2: Residual Analysis (Actual - Predicted vs Predicted)
fig = reg.plot_residuals(y_test_r, reg_preds)
plt.show()


In [ ]:
# Regression Model Comparison Bar Charts
fig = reg.plot_model_comparison(reg_results)
plt.show()


---
## 4. Discriminant Analysis and Classification (LDA & QDA)

We frame the problem as classifying whether a wine is **High Quality** (`quality >= 7`, class 1) or **Normal/Low Quality** (`quality < 7`, class 0).

### Key Theory:
1. **Linear Discriminant Analysis (LDA):**
   - Assumes predictors in each class follow a multivariate normal distribution $\mathcal{N}(\mu_k, \Sigma)$ with a **shared covariance matrix** across all classes ($\Sigma_1 = \Sigma_0 = \Sigma$).
   - Yields a **linear decision boundary**.
   - Parameters to estimate: $p$ means for each class, and 1 shared $p \times p$ covariance matrix.
   - Low variance, less prone to overfitting with modest minority class size.

2. **Quadratic Discriminant Analysis (QDA):**
   - Assumes predictors follow $\mathcal{N}(\mu_k, \Sigma_k)$ with **class-specific covariance matrices**.
   - Yields a **quadratic decision boundary** (allows different spreads/curvatures per class).
   - Parameters to estimate: $K \times \frac{p(p+1)}{2}$ covariance terms.
   - Higher flexibility, but higher variance.

3. **Logistic Regression:**
   - Standard discriminative linear benchmark modeling the log-odds directly without assuming multivariate normality.


In [ ]:
# Check class balance
class_tbl = clf.class_balance(df[BINARY_TARGET])
class_tbl


In [ ]:
# Build classifiers and evaluate on test set with 5-fold CV
clf_models = clf.build_classifiers()
clf_models, clf_results, clf_scores, clf_cms = clf.fit_and_evaluate(
    clf_models, X_train_c, X_test_c, y_train_c, y_test_c
)
clf_results.round(4)


In [ ]:
# Display Confusion Matrices (TP, FP, TN, FN)
fig = clf.plot_confusion_matrices(clf_cms)
plt.show()

cm_table = pd.DataFrame(
    {name: cm.ravel() for name, cm in clf_cms.items()},
    index=["True Neg (TN)", "False Pos (FP)", "False Neg (FN)", "True Pos (TP)"]
).T
cm_table


In [ ]:
# ROC Curves and AUC Comparison
fig = clf.plot_roc_curves(y_test_c, clf_scores)
plt.show()


---
## 5. Comparative Discussion: LDA vs QDA vs Logistic Regression

| Criterion | Linear Discriminant Analysis (LDA) | Quadratic Discriminant Analysis (QDA) | Logistic Regression |
|:---|:---|:---|:---|
| **Underlying Model** | Generative (Multivariate Gaussian) | Generative (Multivariate Gaussian) | Discriminative (Log-odds / Bernoulli) |
| **Covariance Assumption** | Equal covariance ($\Sigma_0 = \Sigma_1$) | Distinct covariance ($\Sigma_0 \neq \Sigma_1$) | No covariance/normality assumption |
| **Decision Boundary** | Linear | Quadratic | Linear |
| **Parameter Count** | Low ($p$ means + 1 shared covariance) | High ($p$ means + $K$ distinct covariances) | Low ($p$ weights + 1 intercept) |
| **Bias-Variance Profile** | Lower variance, higher bias | Higher variance, lower bias | Robust, low variance |
| **Observed Accuracy** | **88.97%** | 85.29% | 87.50% |
| **Observed F1 Score** | **0.4828** | 0.4737 | 0.3462 |
| **Observed ROC-AUC** | **0.8855** | 0.8094 | **0.8860** |

### Key Analytical Insights:
- **Why LDA outperforms QDA here:** High-quality wines account for only ~13.5% of the data. QDA must estimate separate $11 \times 11$ covariance matrices for both classes, leading to higher estimation variance on the small minority class. LDA's pooled covariance constraint acts as effective regularization.
- **Why Logistic Regression has lower recall at 0.5 threshold:** Default Logistic Regression predicts probabilities and defaults to 0.5 cut-off, causing it to predict high quality conservatively (high precision 60%, but recall 24.3%). Both LDA and QDA provide better sensitivity (recall ~38-49%) for detecting premium wines.


---
## 6. Summary of Key Findings

1. **Exploratory Data Analysis:**
   - Red wine quality ratings are centered around 5 and 6 (approx. 82.5% of wines).
   - Alcohol has the strongest positive correlation with quality ($r = +0.48$), whereas volatile acidity has the strongest negative correlation ($r = -0.40$).
2. **Regression Modeling:**
   - Multiple Linear Regression achieves a test RMSE of **0.6565** and an $R^2$ of **0.3915**, significantly outperforming the baseline mean predictor ($RMSE = 0.8443$).
   - Standardized coefficients identify **alcohol (+0.334)**, **volatile acidity (-0.178)**, and **sulphates (+0.148)** as the dominant drivers of quality.
3. **Discriminant Analysis & Classification:**
   - **LDA** delivers the highest overall classification accuracy (**88.97%**) and highest F1-score (**0.4828**) on the test set, achieving an ROC-AUC of **0.8855**.
   - The linear boundary assumption of LDA is well suited for this feature set and protects against the overfitting risks of QDA.
